# Wet-cooled ORC case: PUE, EUE, CUE and WUE
Results for the wet-cooled ORC case only. The base case is handled in its own section, so this notebook uses only the few numbers it needs from the base case (set in Step 1).

**How to use:** save this notebook in the same folder as the latest ORC model notebook, choose a Python kernel that has CoolProp installed (top right), then **Run All**.

## Setup

In [ ]:
import json, os, io, contextlib, warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from CoolProp.CoolProp import PropsSI as PS
from CoolProp.HumidAirProp import HAPropsSI


## Step 1: Inputs
Every number here needs a cited source in the report. The base-case inputs come from the base-case section of the report.

In [ ]:
# --- Facility (brief) ---
P_IT        = 3000.0                    # kW, IT load
HOURS_YR    = 8000                      # h/yr at full load (zero load otherwise)
RECOVERABLE = 0.80                      # fraction of IT electricity captured as heat in the liquid loop
PUE_base    = 1.20                      # ASSUMPTION shared with the base case (brief's starting value)

# --- Emissions and water ---
EF_grid     = 706.2 * 0.45359237 / 1000 # kg CO2e per kWh: eGRID2023 Rev.2, AZNM (CO2e, not CO2 only)
DIRECT_EMISSIONS_t = 0.0                # t CO2e/yr from on-site fuel: none
EWIF        = 4.52                      # L of water per kWh of grid electricity (LBNL 2024, indirect)
COC         = float('inf')              # team assumption: no mineral build-up, so no blowdown
Q_reuse_kWh = 0.0                       # no heat user: the ORC's electricity is not Q_reuse (module guidance)
ORC_ELEC_FACTOR = 1.0                   # applied to ORC gross generation; 1.0 = no separate generator loss

# --- Base-case inputs (from the base-case section; do not change them here) ---
BASE_HEAT_REJECTED_MWh = 19200.0        # heat rejected by the base-case towers, MWh/yr
BASE_TOWER_PARTS_MWh = {'circulation pumps': 69.7, 'fans': 17.9, 'spray pumps': 13.4}   # MWh/yr
BASE_TOWER_ENERGY_MWh = sum(BASE_TOWER_PARTS_MWh.values())     # 101.0 MWh/yr of heat-rejection energy

# --- Method choice ---
# How the 8,000 operating hours are spread over the weather bins:
#   'orc_notebook' : the ORC model notebook's own 8,000-hour block (used for uniformity with the ORC model owner)
#   'even'         : every bin scaled by 8,000/8,760
HOURS_METHOD = 'orc_notebook'

ORC_NOTEBOOK = "ORC_optimiser_1.ipynb"  # latest ORC model notebook, saved in the same folder as this notebook


## Step 2: Read the ORC model results
Runs the ORC model notebook (cells that fail are skipped and listed) and takes its weather bins and functions.

In [ ]:
assert os.path.exists(ORC_NOTEBOOK), "Cannot find " + ORC_NOTEBOOK + " in the folder " + os.getcwd()

# A notebook is a JSON file. Read it, then run its code cells one by one in a separate workspace.
orc_model = {'np': np, 'pd': pd, 'PS': PS, 'HAPropsSI': HAPropsSI}     # names the notebook expects
with open(ORC_NOTEBOOK, encoding='utf-8') as f:
    notebook = json.load(f)

skipped = []
for number, cell in enumerate(notebook['cells']):
    if cell['cell_type'] != 'code':
        continue
    source = ''.join(cell['source'])
    source = '\n'.join(line for line in source.split('\n') if not line.strip().startswith(('%', '!')))
    source = source.replace('plt.show()', 'pass')            # do not display the notebook's plots here
    try:
        with contextlib.redirect_stdout(io.StringIO()), warnings.catch_warnings():
            warnings.simplefilter('ignore')
            exec(source, orc_model)
    except Exception as error:
        skipped.append((number, repr(error)))
plt.close('all')
if skipped:
    print("ORC model notebook: cells skipped because of errors (cell number, error):")
    for item in skipped:
        print("  ", item)

needed_names = ['bin_results_df', 'calculate_system_performance', 'wet_tower', 'P_atm', 'T_approach', 'T_range',
                'fl', 'SC', 'SH', 'eta_t', 'eta_p', 'm_w', 'cpw', 'T_DC_out', 'PP']
if HOURS_METHOD == 'orc_notebook':
    needed_names.append('operating_df')
for name in needed_names:
    assert name in orc_model, "The ORC model notebook does not define '" + name + "' (skipped cells: " + str(skipped) + ")"

bin_results_df = orc_model['bin_results_df']
calculate_system_performance = orc_model['calculate_system_performance']
wet_tower = orc_model['wet_tower']
T_DC_return = 40.0 + 273.15
print("Weather bins:", len(bin_results_df), "| bins with no valid ORC design:", int(bin_results_df['Optimal_T_evap_C'].isna().sum()))


## Step 2b: Turbine and feed-pump work
The ORC model returns only net work. This recalculates the turbine work and the ORC feed pump work from the model's own constants, and checks that they reproduce its net work.

In [ ]:
def orc_work_split(T_evap_K, T_cond_K):
    fl, SC, SH = orc_model['fl'], orc_model['SC'], orc_model['SH']
    eta_t, eta_p = orc_model['eta_t'], orc_model['eta_p']
    P_e = PS('P', 'T', T_evap_K, 'Q', 1, fl)                     # evaporator pressure
    P_c = PS('P', 'T', T_cond_K, 'Q', 1, fl)                     # condenser pressure
    T1 = T_cond_K - SC                                           # pump inlet temperature
    h1 = PS('H', 'T', T1, 'Q', 0, fl)                            # pump inlet enthalpy
    s1 = PS('S', 'T', T1, 'Q', 0, fl)
    h2 = h1 + (PS('H', 'P', P_e, 'S', s1, fl) - h1) / eta_p      # pump outlet enthalpy
    h3 = PS('H', 'T', T_evap_K, 'Q', 0, fl)                      # saturated liquid at the evaporating temperature
    h4 = PS('H', 'T', T_evap_K + SH, 'P', P_e, fl)               # turbine inlet enthalpy
    s4 = PS('S', 'T', T_evap_K + SH, 'P', P_e, fl)
    h5 = h4 - eta_t * (h4 - PS('H', 'P', P_c, 'S', s4, fl))      # turbine outlet enthalpy
    Q_boil = orc_model['m_w'] * orc_model['cpw'] * (orc_model['T_DC_out'] - (T_evap_K + orc_model['PP']))
    m_wf = Q_boil / (h4 - h3)                                    # working fluid flow, kg/s
    return m_wf * (h4 - h5) / 1e3, m_wf * (h2 - h1) / 1e3        # turbine work, feed pump work (kW)


## Step 2c: One row per weather bin
For each bin, calls the ORC model's own function with the best evaporating temperature it found and the wet-bulb it used (corrected where the weather data was impossible). A bin with no valid ORC design is treated as ORC off: all the loop heat goes to the trim tower.

In [ ]:
P_atm, T_approach, T_range = orc_model['P_atm'], orc_model['T_approach'], orc_model['T_range']
Q_loop = orc_model['m_w'] * orc_model['cpw'] * (orc_model['T_DC_out'] - T_DC_return)     # W, heat in the 60/40 C loop

rows = []
for i, r in bin_results_df.iterrows():
    T_db_K = r['T_db_bin'] + 273.15
    T_wb_K = r['T_wb_used_C'] + 273.15 if 'T_wb_used_C' in bin_results_df.columns else r['T_wb_bin'] + 273.15
    row = {'T_db_bin': r['T_db_bin'], 'TMY_hours': r['TMY_hours']}
    if pd.notna(r['Optimal_T_evap_C']):
        res = calculate_system_performance(r['Optimal_T_evap_C'], r['T_cond_C'], T_db_K, T_wb_K, P_atm, T_approach, T_range)
        gross_kW, pump_kW = orc_work_split(r['Optimal_T_evap_C'] + 273.15, r['T_cond_C'] + 273.15)
        assert abs((gross_kW - pump_kW) - res['W_net_orc'] / 1e3) < 1e-6 * max(1.0, gross_kW), \
            "orc_work_split does not match the ORC model's net work: the ORC model has changed"
        row.update({
            'W_gross_kW': gross_kW, 'W_pump_orc_kW': pump_kW, 'W_net_orc_kW': res['W_net_orc'] / 1e3,
            'evap_orc_kgs': res['water_ORC_case_L_h'] / 3600,
            'Q_in_kW': res['Q_in'] / 1e3, 'Q_cond_kW': res['Q_cond'] / 1e3, 'Q_trim_kW': res['Q_trim'] / 1e3})
    else:
        tower = wet_tower(Q_loop, T_db_K, T_wb_K, P_atm, T_approach, T_range)   # ORC off: one tower carries everything
        row.update({
            'W_gross_kW': 0.0, 'W_pump_orc_kW': 0.0, 'W_net_orc_kW': 0.0,
            'evap_orc_kgs': tower['evap_kg_s'],
            'Q_in_kW': 0.0, 'Q_cond_kW': 0.0, 'Q_trim_kW': Q_loop / 1e3})
    rows.append(row)
bins = pd.DataFrame(rows)

# Operating hours
if HOURS_METHOD == 'even':
    bins['hours'] = bins['TMY_hours'] * HOURS_YR / bins['TMY_hours'].sum()
elif HOURS_METHOD == 'orc_notebook':
    hours_by_bin = orc_model['operating_df'].groupby('T_db_bin')['TMY_hours'].sum()
    bins['hours'] = bins['T_db_bin'].map(hours_by_bin).fillna(0.0)
else:
    raise ValueError("HOURS_METHOD must be 'even' or 'orc_notebook'")
assert abs(bins['hours'].sum() - HOURS_YR) < 1e-6, "operating hours must add up to HOURS_YR"
print("Hours method:", HOURS_METHOD, "| total operating hours:", round(bins['hours'].sum(), 1))


## Step 3: Annual totals
Power in kW multiplied by hours gives energy in kWh. Covers energy, heat and water.

In [ ]:
# Energy
E_IT = P_IT * HOURS_YR                                    # kWh, IT energy
E_DC_base = PUE_base * E_IT                               # kWh, base-case facility energy (shared assumption)

# ORC electricity: gross generation, the ORC feed pump (a parasitic load) and net output are kept separate
E_gross = ORC_ELEC_FACTOR * (bins['W_gross_kW'] * bins['hours']).sum()   # kWh, gross ORC generation
E_pump  = (bins['W_pump_orc_kW'] * bins['hours']).sum()                  # kWh, ORC feed pump electricity
E_net   = E_gross - E_pump                                               # kWh, net ORC electrical output

# Heat
loop_heat          = RECOVERABLE * E_IT                                  # kWh of heat in the liquid loop
heat_to_orc        = (bins['Q_in_kW'] * bins['hours']).sum()             # kWh taken by the ORC
heat_rejected_orc  = ((bins['Q_cond_kW'] + bins['Q_trim_kW']) * bins['hours']).sum()   # kWh, towers, ORC case

# Cooling tower electricity (fans and pumps): the base-case section's tower model is applied to both cases
base_intensity = BASE_TOWER_ENERGY_MWh / BASE_HEAT_REJECTED_MWh          # kWh of tower electricity per kWh of heat rejected
E_tower_base = BASE_TOWER_ENERGY_MWh * 1e3                               # kWh, base case (base-case section)
E_tower_orc  = base_intensity * heat_rejected_orc                        # kWh, ORC case (condenser + trim towers)
tower_change = E_tower_orc - E_tower_base                                # kWh, change in tower fan + pump energy

# Facility energy and grid imports
par_change = tower_change + E_pump      # kWh, total change in parasitic load (towers + ORC feed pump)
E_DC_orc   = E_DC_base + par_change     # kWh, total facility energy, ORC case (NOT reduced by the ORC's generation)
imports_orc = E_DC_orc - E_gross        # kWh bought from the grid, ORC case

# Water (1 kg of water = 1 litre)
evap_orc_L = (bins['evap_orc_kgs'] * bins['hours']).sum() * 3600         # L/yr consumed (evaporated)
blowdown_L = evap_orc_L / (COC - 1)                                      # L/yr drained (zero when COC is infinite)
withdrawn_L = evap_orc_L + blowdown_L                                    # L/yr taken in (makeup)


## Step 3b: Base-case parameters in use
The numbers taken from the base-case section, and the values derived from them, checked against what that section states.

In [ ]:
print("Base-case parameters in use")
print("  Heat rejected by the base-case towers:   %s MWh/yr" % format(BASE_HEAT_REJECTED_MWh, ',.0f'))
for part, value in BASE_TOWER_PARTS_MWh.items():
    print("  Tower electricity, %-18s %6.1f MWh/yr" % (part + ':', value))
print("  Tower electricity, total:                %6.1f MWh/yr" % BASE_TOWER_ENERGY_MWh)
print("  Per kWh of heat rejected:                %.4f kWh/kWh   (base-case section: 0.0053)" % base_intensity)
print("  Share of IT energy:                      %.2f %%         (base-case section: 0.42 %%)" % (BASE_TOWER_ENERGY_MWh * 1e3 / E_IT * 100))
print("  Constant power over %d h:              %.2f kW" % (HOURS_YR, BASE_TOWER_ENERGY_MWh * 1e3 / HOURS_YR))
assert abs(base_intensity - 0.0053) < 5e-5, "tower electricity per kWh of heat rejected no longer matches the base-case section"
assert abs(BASE_TOWER_ENERGY_MWh * 1e3 / E_IT * 100 - 0.42) < 0.005, "tower electricity as a share of IT energy no longer matches the base-case section"
assert abs(BASE_TOWER_ENERGY_MWh - 101.0) < 1e-9, "the parts of the base-case tower electricity no longer add up to 101.0 MWh"
print("\nTower electricity: base case %.1f MWh/yr | ORC case (condenser and trim towers) %.1f MWh/yr" % (E_tower_base / 1e3, E_tower_orc / 1e3))


## Step 4: The indicators
PUE, EUE, CUE, WUE (site and source), and the avoided emissions relative to the base case (reported separately from CUE).

In [ ]:
PUE_orc = E_DC_orc / E_IT                                        # PUE = E_DC / E_IT
EUE_orc = (E_IT + Q_reuse_kWh) / E_DC_orc                        # EUE_IT = (E_IT + Q_reuse) / E_DC

emissions_orc = (EF_grid * imports_orc) / 1000 + DIRECT_EMISSIONS_t      # t CO2e/yr
CUE_orc = emissions_orc * 1000 / E_IT                            # kg CO2e per kWh of IT energy

WUE_site_orc   = evap_orc_L / E_IT                               # L/kWh IT
WUE_source_orc = (evap_orc_L + EWIF * imports_orc) / E_IT        # adds the water used to generate the grid electricity

# Avoided emissions: the base case has no ORC, so its grid imports equal its facility energy
emissions_base_ref = (EF_grid * E_DC_base) / 1000 + DIRECT_EMISSIONS_t
avoided_emissions = emissions_base_ref - emissions_orc           # t CO2e/yr, separate from CUE


## Step 5: Results table (wet-cooled ORC case)

In [ ]:
wet = {
    'E_IT (MWh)': E_IT / 1e3,
    'E_DC (MWh)': E_DC_orc / 1e3,
    'Cooling towers: change in fan + pump energy (MWh)': tower_change / 1e3,
    'ORC feed pump, parasitic (MWh)': E_pump / 1e3,
    'ORC gross generation (MWh)': E_gross / 1e3,
    'ORC net electrical output (MWh)': E_net / 1e3,
    'Net grid imports (MWh)': imports_orc / 1e3,
    'IT heat generated (MWh)': E_IT / 1e3,
    'Recoverable heat in loop (MWh)': loop_heat / 1e3,
    'Heat taken by ORC (MWh)': heat_to_orc / 1e3,
    'Heat reused, Q_reuse (MWh)': Q_reuse_kWh / 1e3,
    'Heat rejected by towers (MWh)': heat_rejected_orc / 1e3,
    'PUE = E_DC/E_IT': PUE_orc,
    'EUE_IT = (E_IT+Q_reuse)/E_DC': EUE_orc,
    'CUE = G_DC/E_IT (kg CO2e/kWh IT)': CUE_orc,
    'WUE_site (L/kWh IT)': WUE_site_orc,
    'WUE_source (L/kWh IT)': WUE_source_orc,
    'Emissions (t CO2e/yr)': emissions_orc,
    'Water consumed = evaporation (m3/yr)': evap_orc_L / 1e3,
    'Blowdown (m3/yr)': blowdown_L / 1e3,
    'Water withdrawn (m3/yr)': withdrawn_L / 1e3,
    'Avoided emissions vs the base case (t CO2e/yr)': avoided_emissions,
}
results = pd.DataFrame({'Wet-cooled ORC': wet})
print(results.round(4).to_string())

print("\nFor the report's comparison table (wet-cooled column):")
print("  Annual facility energy used:          %s MWh/yr" % format(E_DC_orc / 1e3, ',.0f'))
print("  Annual heat rejected:                 %s MWh/yr" % format(heat_rejected_orc / 1e3, ',.0f'))
print("  Annual heat used (Q_reuse):           0 MWh/yr (no heat user)")
print("  Annual electricity generated:         %s MWh/yr gross, %s MWh/yr net of the ORC feed pump" % (format(E_gross / 1e3, ',.0f'), format(E_net / 1e3, ',.0f')))
print("  Annual CO2e emissions:                %s t/yr" % format(emissions_orc, ',.0f'))
print("  Annual water consumption:             %s L/yr" % format(evap_orc_L, ',.0f'))


## Step 5a: Reconciliation of facility energy and grid imports
Shows how facility energy is built up and how net grid imports follow from it. Each parasitic load appears once, and the ORC's generation is deducted only at the end.

In [ ]:
reconciliation = pd.DataFrame({'Wet-cooled ORC (MWh/yr)': [
        E_DC_base / 1e3, tower_change / 1e3, E_pump / 1e3, E_DC_orc / 1e3, -E_gross / 1e3, imports_orc / 1e3, E_net / 1e3,
        E_tower_base / 1e3, E_tower_orc / 1e3]},
    index=['Base-case facility energy, PUE_base x E_IT',
           '+ Cooling towers: change in fan + pump energy',
           '+ ORC feed pump, parasitic',
           '= Facility energy, E_DC',
           '- ORC gross generation',
           '= Net grid imports',
           'Memo: ORC net electrical output = gross - feed pump',
           'Memo: tower electricity, base case (base-case section)',
           'Memo: tower electricity, ORC case (condenser + trim towers)'])
print(reconciliation.round(1).to_string())


## Step 5b: Reconciliation check
Stops with a message if any quantity is counted twice or the numbers do not reconcile.

In [ ]:
r = results['Wet-cooled ORC']
tol = 1e-6
tower_row = 'Cooling towers: change in fan + pump energy (MWh)'
pump_row  = 'ORC feed pump, parasitic (MWh)'
gross_row = 'ORC gross generation (MWh)'
net_row   = 'ORC net electrical output (MWh)'

# 1. Facility energy = base-case energy + change in tower energy + ORC feed pump (the generation is NOT deducted here)
assert abs(r['E_DC (MWh)'] - (E_DC_base / 1e3 + r[tower_row] + r[pump_row])) < tol, \
    "Facility energy does not equal base-case energy + change in tower energy + ORC feed pump"
# 2. Net output = gross generation - ORC feed pump
assert abs(r[net_row] - (r[gross_row] - r[pump_row])) < tol, "ORC net output does not equal gross generation minus the feed pump"
# 3. Net grid imports = facility energy - gross generation
assert abs(r['Net grid imports (MWh)'] - (r['E_DC (MWh)'] - r[gross_row])) < tol, \
    "Net grid imports do not equal facility energy minus gross generation"
# 4. The same imports from the other side: base-case energy + change in tower energy - net output (feed pump counted once)
assert abs(r['Net grid imports (MWh)'] - (E_DC_base / 1e3 + r[tower_row] - r[net_row])) < tol, \
    "Net grid imports do not equal base-case energy + change in tower energy - ORC net output"
# 5. CUE is based on the emissions of purchased electricity only, not reduced again by the avoided emissions
assert abs(r['CUE = G_DC/E_IT (kg CO2e/kWh IT)'] * r['E_IT (MWh)'] - r['Emissions (t CO2e/yr)']) < tol, \
    "CUE does not match the emissions of the purchased electricity"
# 6. Avoided emissions = emission factor x saving in grid imports
assert abs(r['Avoided emissions vs the base case (t CO2e/yr)'] - EF_grid * (E_DC_base / 1e3 - r['Net grid imports (MWh)'])) < tol, \
    "Avoided emissions do not match the emission factor times the saving in grid imports"
print("Reconciliation check passed: gross generation, the ORC feed pump, tower energy and net grid imports are each counted once.")


## Step 5c: Check against the formulas written in the report
Re-calculates every indicator exactly as the report's equations are written, with separate variable names, and confirms the results table holds the same values.

In [ ]:
# Report equations (data center loop pump is in both cases, so it cancels and is left out of both parasitic terms)
E_IT_r        = P_IT * HOURS_YR
E_DC_base_r   = PUE_base * E_IT_r                                  # E_DC,base = PUE_base x E_IT
E_par_ORC_r   = E_tower_orc                                        # E_parasitic,ORC: cooling tower fan + pump energy, ORC case
E_par_base_r  = E_tower_base                                       # E_parasitic,base: cooling tower fan + pump energy, base case
E_pump_ORC_r  = E_pump                                             # E_pump,ORC: ORC working-fluid feed pump
dE_par_r      = E_par_ORC_r + E_pump_ORC_r - E_par_base_r          # dE_parasitic,wet
E_DC_ORC_r    = E_DC_base_r + dE_par_r                             # E_DC,ORC,wet = E_DC,base + dE_parasitic,wet
PUE_r         = E_DC_ORC_r / E_IT_r                                # PUE_ORC,wet = E_DC,ORC,wet / E_IT
EUE_r         = (E_IT_r + Q_reuse_kWh) / E_DC_ORC_r                # EUE_IT = (E_IT + Q_reuse) / E_DC, with Q_reuse = 0
E_import_r    = E_DC_ORC_r - E_gross                               # E_net import = E_DC,ORC,wet - E_gross,ORC
CUE_r         = (E_import_r * EF_grid) / E_IT_r                    # CUE = E_net import x CI / E_IT (direct emissions = 0)
V_cons_r      = evap_orc_L                                         # V_cons: water evaporated in the two cooling towers
WUE_site_r    = V_cons_r / E_IT_r                                  # WUE_site = V_cons / E_IT
WUE_source_r  = (V_cons_r + E_import_r * EWIF) / E_IT_r            # WUE_source = (V_cons + E_net import x EWIF) / E_IT
avoided_r     = (E_DC_base_r - E_import_r) * EF_grid / 1000        # avoided emissions = (E_DC,base - E_net import) x CI

table = [
    ('E_DC,ORC,wet (MWh)',        E_DC_ORC_r / 1e3,   r['E_DC (MWh)']),
    ('PUE_ORC,wet',               PUE_r,              r['PUE = E_DC/E_IT']),
    ('EUE_IT',                    EUE_r,              r['EUE_IT = (E_IT+Q_reuse)/E_DC']),
    ('E_net import (MWh)',        E_import_r / 1e3,   r['Net grid imports (MWh)']),
    ('CUE (kg CO2e/kWh IT)',      CUE_r,              r['CUE = G_DC/E_IT (kg CO2e/kWh IT)']),
    ('WUE_site (L/kWh IT)',       WUE_site_r,         r['WUE_site (L/kWh IT)']),
    ('WUE_source (L/kWh IT)',     WUE_source_r,       r['WUE_source (L/kWh IT)']),
    ('Avoided emissions (t/yr)',  avoided_r,          r['Avoided emissions vs the base case (t CO2e/yr)']),
]
check_table = pd.DataFrame(table, columns=['Quantity', "Report's formula", 'Results table'])
check_table['Difference'] = check_table["Report's formula"] - check_table['Results table']
print(check_table.round(6).to_string(index=False))
for name, from_report, from_table in table:
    assert abs(from_report - from_table) < 1e-6 * max(1.0, abs(from_table)), name + ": the report's formula and the results table disagree"
print("\nFormula check passed: the values in the results table are exactly what the report's equations give.")


## Step 6: Plots (wet-cooled ORC case)
Four plots of the ORC case: how the ORC's output changes with the weather, how the loop heat is shared between the two towers, what changes net grid imports, and where the water in WUE_source comes from.

In [ ]:
# Plot 1: ORC electrical power against the outdoor (dry-bulb) temperature
gross_kW = ORC_ELEC_FACTOR * bins['W_gross_kW']
net_kW = gross_kW - bins['W_pump_orc_kW']

fig1, ax1 = plt.subplots(figsize=(9, 5))
ax2 = ax1.twinx()
ax2.bar(bins['T_db_bin'], bins['hours'], width=1.6, color='lightgray', label='Operating hours in the bin')
ax2.set_ylabel('Operating hours in the bin (h)')
ax1.set_zorder(ax2.get_zorder() + 1)                  # draw the lines above the bars
ax1.patch.set_visible(False)
ax1.plot(bins['T_db_bin'], gross_kW, marker='o', label='Gross generation')
ax1.plot(bins['T_db_bin'], net_kW, marker='s', label='Net output (after the ORC feed pump)')
ax1.plot(bins['T_db_bin'], bins['W_pump_orc_kW'], marker='^', label='ORC feed pump')
ax1.set_xlabel('Dry-bulb temperature bin (\u00b0C)')
ax1.set_ylabel('ORC electrical power (kW)')
ax1.set_title('ORC electrical power against outdoor temperature')
ax1.grid(True)
ax1.legend(loc='upper right')
plt.tight_layout()
plt.savefig("orc_power_by_temperature.png", dpi=150)
plt.show()

# Plot 2: how the loop heat is shared between the two cooling towers
fig2, ax = plt.subplots(figsize=(9, 5))
ax.bar(bins['T_db_bin'], bins['Q_cond_kW'], width=1.6, label='Condenser tower (heat rejected by the ORC)')
ax.bar(bins['T_db_bin'], bins['Q_trim_kW'], width=1.6, bottom=bins['Q_cond_kW'], label='Trim tower (rest of the loop heat)')
ax.set_xlabel('Dry-bulb temperature bin (\u00b0C)')
ax.set_ylabel('Heat rejected while operating (kW)')
ax.set_title('Heat rejected by the two cooling towers')
ax.grid(True, axis='y')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig("tower_heat_split.png", dpi=150)
plt.show()

# Plot 3: what changes net grid imports compared with the base-case facility energy (MWh/yr)
labels = ['ORC gross generation', 'ORC feed pump', 'Cooling towers: change in energy', 'Net change in grid imports']
changes = [-E_gross / 1e3, E_pump / 1e3, tower_change / 1e3, (imports_orc - E_DC_base) / 1e3]
colours = ['tab:blue' if v < 0 else 'tab:orange' for v in changes]
fig3, ax3 = plt.subplots(figsize=(9, 4.5))
bars = ax3.barh(labels, changes, color=colours)
ax3.axvline(0, color='black', linewidth=0.8)
ax3.invert_yaxis()
ax3.set_xlim(min(changes) * 1.25, max(max(changes) * 3, 60))
for bar, value in zip(bars, changes):
    ax3.text(value, bar.get_y() + bar.get_height() / 2, '  ' + format(value, '+,.1f'), va='center', ha='left' if value >= 0 else 'right')
ax3.set_xlabel('Change in net grid imports (MWh/yr): blue lowers imports, orange raises them')
ax3.set_title('What changes net grid imports')
ax3.grid(True, axis='x')
plt.tight_layout()
plt.savefig("grid_import_changes.png", dpi=150)
plt.show()

# Plot 4: where the water in WUE_source comes from (L per kWh of IT energy)
site_part = WUE_site_orc
grid_part = EWIF * imports_orc / E_IT
fig4, ax4 = plt.subplots(figsize=(9, 2.8))
ax4.barh(['WUE_source'], [site_part], label='Cooling-tower evaporation (WUE_site)')
ax4.barh(['WUE_source'], [grid_part], left=[site_part], label='Water used to generate the grid electricity')
ax4.text(site_part / 2, 0, format(site_part, '.2f'), ha='center', va='center', color='white')
ax4.text(site_part + grid_part / 2, 0, format(grid_part, '.2f'), ha='center', va='center', color='white')
ax4.set_xlabel('L per kWh of IT energy (total = %.2f)' % (site_part + grid_part))
ax4.set_title('Where the water in WUE_source comes from')
ax4.legend(loc='upper center', bbox_to_anchor=(0.5, -0.45), ncol=2)
plt.tight_layout()
plt.savefig("wue_source_breakdown.png", dpi=150)
plt.show()
